# Experiment 1b — Authority direction from conflict pairs

Uses the 4-trace contrastive design on evolved conflict pairs to find the authority
direction from behavioral differences (system-following vs user-following), not structural position.

Requires a GPU (A100 recommended). Set MODEL_KEY in the config cell, then Run All.

In [ ]:
# Install
import os, sys, importlib, site
REPO_DIR = "/content/Mech_spoof"

if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/ChuloIva/Mech_spoof.git {REPO_DIR}

!pip install -q -e {REPO_DIR}

os.environ["MECH_SPOOF_ROOT"] = REPO_DIR

src_path = f"{REPO_DIR}/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()
site.main()

import mech_spoof
print("mech_spoof loaded from:", mech_spoof.__file__)

In [ ]:
# Auth + Drive
import os, getpass
from google.colab import drive, userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
drive.mount("/content/drive")

In [ ]:
# Config (EDIT ME)
from pathlib import Path
from mech_spoof.configs import MODEL_CONFIGS

MODEL_KEY = "qwen"            # qwen | llama3 | mistral | gemma | phi3 | gemma_small
DRIVE_ROOT = Path("/content/drive/MyDrive/mech_spoof_results")

slug = MODEL_CONFIGS[MODEL_KEY].slug
OUT_DIR = DRIVE_ROOT / slug / "exp1b_authority_conflict"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Activation caches on local Colab disk — fast I/O, not persisted across sessions.
ACT_CACHE_DIR = Path("/content/act_cache") / slug / "exp1b_authority_conflict"
ACT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

EXP1_DIR = DRIVE_ROOT / slug / "exp1_authority"  # for cross-check (optional)

print(f"Model: {MODEL_KEY} ({MODEL_CONFIGS[MODEL_KEY].hf_id})")
print(f"OUT_DIR       = {OUT_DIR}")
print(f"ACT_CACHE_DIR = {ACT_CACHE_DIR}")
print(f"EXP1_DIR      = {EXP1_DIR} (exists={EXP1_DIR.exists()})")

In [ ]:
# Run experiment 1b
from mech_spoof.experiments.exp1b_authority_conflict import run_experiment_1b

result = run_experiment_1b(
    MODEL_KEY,
    OUT_DIR,
    exp1_dir=EXP1_DIR if EXP1_DIR.exists() else None,
    activation_cache_dir=ACT_CACHE_DIR,
    batch_size=4,       # bump to 8-16 on A100 80GB
    max_length=2048,
    max_response_chars=3000,
)
print(f"\nBest layer: {result.best_layer}")
print(f"Best accuracy: {result.best_accuracy:.4f}")

## Results

In [ ]:
# Per-layer accuracy & AUROC — one line per extraction position
import matplotlib.pyplot as plt
import numpy as np

positions = result.positions
n_layers = result.n_layers
layers = list(range(n_layers))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for pos in positions:
    accs = [result.accuracies[pos][l] for l in layers]
    aurs = [result.aurocs[pos][l] for l in layers]
    ax1.plot(layers, accs, marker="o", markersize=3, label=pos)
    ax2.plot(layers, aurs, marker="s", markersize=3, label=pos)

for ax, title, ylabel in [
    (ax1, f"{MODEL_KEY}: exp1b per-layer accuracy", "Probe accuracy"),
    (ax2, f"{MODEL_KEY}: exp1b per-layer AUROC", "AUROC"),
]:
    ax.axhline(0.5, color="gray", ls="--", lw=0.8, label="chance")
    ax.axvline(result.best_layer, color="red", ls=":", alpha=0.6,
               label=f"best: L{result.best_layer} ({result.best_position})")
    ax.set_xlabel("Layer"); ax.set_ylabel(ylabel); ax.set_ylim(0.4, 1.05)
    ax.set_title(title); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / "layer_accuracy.png", dpi=120)
plt.show()

In [ ]:
# Probe vs diff-in-means agreement (per position)
fig, ax = plt.subplots(figsize=(9, 4))
for pos in positions:
    cos = [result.probe_vs_dim_cosine[pos][l] for l in layers]
    ax.plot(layers, cos, marker=".", label=pos)
ax.set_xlabel("Layer"); ax.set_ylabel("Cosine similarity")
ax.set_title(f"{MODEL_KEY}: probe dir vs diff-in-means (per position)")
ax.set_ylim(-0.1, 1.1); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "probe_vs_dim_cosine.png", dpi=120)
plt.show()

In [ ]:
# Per macro-axis breakdown at global best (pos, layer)
breakdown = result.macro_axis_breakdown
print(f"Best: pos={result.best_position}  layer={result.best_layer}  "
      f"overall acc={result.best_accuracy:.4f}\n")
print(f"{'Axis':>15s}  {'N':>5s}  {'Acc':>6s}  {'AUROC':>6s}")
print("-" * 38)
for axis in sorted(breakdown, key=lambda a: -breakdown[a]["accuracy"]):
    d = breakdown[axis]
    print(f"{axis:>15s}  {d['n']:5d}  {d['accuracy']:6.3f}  {d['auroc']:6.3f}")

In [ ]:
# Perplexity distribution (sys-following vs user-following)
print("PPL stats:")
for k, v in result.ppl_stats.items():
    print(f"  {k:30s}  {v:.3f}")

import numpy as np
# Reload from saved arrays so we can show even after session restart.
with np.load(OUT_DIR / "arrays.npz") as data:
    sys_ppl = data["ppl_sys_following"]
    usr_ppl = data["ppl_usr_following"]

clip = np.nanpercentile(np.concatenate([sys_ppl, usr_ppl]), 99)
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(np.clip(sys_ppl, None, clip), bins=60, alpha=0.55, label="sys-following")
ax.hist(np.clip(usr_ppl, None, clip), bins=60, alpha=0.55, label="usr-following")
ax.set_xlabel("Perplexity (clipped at p99)"); ax.set_ylabel("Count")
ax.set_title(f"{MODEL_KEY}: response PPL under forced alignment")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "ppl_hist.png", dpi=120)
plt.show()

# Optional cross-check with exp1 structural direction at the best position
if result.exp1_cosine_agreement is not None:
    pos = result.best_position
    cos_line = [result.exp1_cosine_agreement[pos].get(l, float("nan")) for l in layers]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(layers, cos_line, marker=".", color="purple")
    ax.axhline(0, color="gray", ls="--", lw=0.8)
    ax.set_xlabel("Layer"); ax.set_ylabel("Cosine sim")
    ax.set_title(f"{MODEL_KEY}: exp1b @ {pos} vs exp1 structural direction")
    ax.set_ylim(-1.1, 1.1)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "exp1b_vs_exp1_cosine.png", dpi=120)
    plt.show()

In [ ]:
# Summary
print(f"Model:          {result.model_key}")
print(f"Layers:         {result.n_layers}")
print(f"d_model:        {result.d_model}")
print(f"Pairs:          {result.n_pairs}")
print(f"Traces:         {result.n_system_following + result.n_user_following}"
      f" ({result.n_system_following} sys + {result.n_user_following} usr)")
print(f"Positions:      {list(result.positions)}")
print(f"Best:           pos={result.best_position}  layer={result.best_layer}"
      f"  acc={result.best_accuracy:.4f}")
print(f"\nResults saved to: {OUT_DIR}")
print(f"  - result.json     : per-layer × per-position metrics")
print(f"  - arrays.npz      : probe / dim directions, accs, PPL arrays")
print(f"  - prompts.csv     : per-trace metadata (for Streamlit viewer)")
print(f"  - probes_by_position.pkl")
print(f"\nTo explore locally:")
print(f"  pip install -e '.[viz]'")
print(f"  streamlit run streamlit_apps/exp1b_viewer.py -- --results <path-to-OUT_DIR>")